In [1]:
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import numpy as np
import os

In [2]:
from pace import NullComm

2026-03-24 18:14:12|INFO|rank 0|ndsl.logging:Literal precision: 64
2026-03-24 18:14:12|INFO|rank 0|ndsl.logging:Constant selected: ConstantVersions.UFS


In [3]:
comm = NullComm(0,6,-1.0)

In [9]:
from ndsl import CubedSphereCommunicator, SubtileGridSizer, GridIndexing, QuantityFactory

In [10]:
communicator = CubedSphereCommunicator.from_layout(comm, (1,1))

In [11]:
sizer = SubtileGridSizer.from_tile_params(
    nx_tile=24,
    ny_tile=24,
    nz=79,
    n_halo=3,
    layout=(1,1),
    tile_partitioner=communicator.partitioner.tile,
    tile_rank=communicator.tile.rank,
    backend="numpy",
)

grid_indexing = GridIndexing.from_sizer_and_communicator(
    sizer=sizer, comm=communicator
)
quantity_factory = QuantityFactory(
    sizer, backend="numpy"
)

In [13]:
from ndsl.grid import (
    AngleGridData,
    ContravariantGridData,
    DampingCoefficients,
    DriverGridData,
    GridData,
    HorizontalGridData,
    MetricTerms,
    VerticalGridData,
)
eta_file = "eta79.nc"

In [14]:
metric_terms = MetricTerms(
    quantity_factory=quantity_factory,
    communicator=communicator,
    eta_file=eta_file,
)
horizontal_data = HorizontalGridData.new_from_metric_terms(metric_terms)
vertical_data = VerticalGridData.new_from_metric_terms(metric_terms)
contravariant_data = ContravariantGridData.new_from_metric_terms(metric_terms)
angle_data = AngleGridData.new_from_metric_terms(metric_terms)
grid_data = GridData(
    horizontal_data=horizontal_data,
    vertical_data=vertical_data,
    contravariant_data=contravariant_data,
    angle_data=angle_data,
)

damping_coefficients = DampingCoefficients.new_from_metric_terms(metric_terms)
driver_grid_data = DriverGridData.new_from_metric_terms(metric_terms)

/usr/local/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/usr/local/lib/python3.12/site-packages/ndsl/grid/generation.py:1550: RuntimeWarning: divide by zero encountered in divide
  data=1.0 / self.area.data,
/usr/local/lib/python3.12/site-packages/ndsl/grid/gnomonic.py:681: RuntimeWarning: invalid value encountered in divide
  np.sum(p * q, axis=-1) / np.sqrt(np.sum(p**2, axis=-1) * np.sum(q**2, axis=-1))
/usr/local/lib/python3.12/site-packages/ndsl/grid/gnomonic.py:681: RuntimeWarning: invalid value encountered in scalar divide
  np.sum(p * q, axis=-1) / np.sqrt(np.sum(p**2, axis=-1) * np.sum(q**2, axis=-1))
/usr/local/lib/python3.12/site-packages/ndsl/grid/generation.py:1565: RuntimeWarning: divide by zero encountered in divide
  data=1.0 / self.area_c.data,
/usr/local/lib/python3.12/site-packages/

# Define aquaplanet code

In [16]:
import numpy as np

import ndsl.constants as constants
import ndsl.dsl.gt4py_utils as utils
from ndsl import CubedSphereCommunicator, QuantityFactory
from ndsl.dsl.typing import Float
from ndsl.grid import GridData
from pyfv3.dycore_state import DycoreState
from pyfv3.initialization import init_utils

import math
from dataclasses import fields
from types import SimpleNamespace

import numpy as np

import ndsl.constants as constants
from ndsl.dsl.typing import Float
from ndsl.grid.eta import SURFACE_PRESSURE, compute_eta, vertical_coordinate
from ndsl.grid.gnomonic import (
    get_lonlat_vect,
    get_unit_vector_direction,
    lon_lat_midpoint,
)
from ndsl.logging import ndsl_log
from pyfv3.dycore_state import DycoreState

In [18]:
hydrostatic = False
moist_phys = False

## Aquaplanet code

In [27]:
SURFACE_PRESSURE = Float(1.0e5)  # units of (Pa), from Table VI of DCMIP2016
NHALO = constants.N_HALO_DEFAULT

sample_quantity = grid_data.lat
field_shape = (*sample_quantity.field.shape[0:2], grid_data.ak.data.shape[0])
data_shape = (*sample_quantity.data.shape[0:2], grid_data.ak.data.shape[0])
nx, ny, npz = init_utils.local_compute_size(data_shape)
nz = npz - 1
numpy_state = init_utils.empty_numpy_dycore_state(data_shape)
isc, iec, jsc, jec = init_utils.local_compute_bounds(field_shape)
print(isc, iec, jsc, jec)
print(nx, ny, nz)

hybrid_z = False

# Initializing to values the Fortran does for easy comparison
numpy_state.delp[:] = 1e30
numpy_state.delp[:NHALO, :NHALO] = 0.0
numpy_state.delp[:NHALO, NHALO + ny :] = 0.0
numpy_state.delp[NHALO + nx :, :NHALO] = 0.0
numpy_state.delp[NHALO + nx :, NHALO + ny :] = 0.0
numpy_state.pe[:] = 0.0
numpy_state.pt[:] = 1.0
numpy_state.ua[:] = 1e35
numpy_state.va[:] = 1e35
numpy_state.uc[:] = 1e30
numpy_state.vc[:] = 1e30
numpy_state.w[:] = 1.0e30
numpy_state.delz[:] = 1.0e25
numpy_state.phis[:] = 1.0e25
numpy_state.ps[:] = SURFACE_PRESSURE
eta = np.zeros(npz)
eta_v = np.zeros(npz)
islice, jslice, slice_3d, slice_2d = init_utils.compute_slices(nx, ny)
# Slices with extra buffer points in the horizontal dimension
# to accomodate averaging over shifted calculations on the grid
_, _, slice_3d_buffer, slice_2d_buffer = init_utils.compute_slices(nx + 1, ny + 1)

init_utils.setup_pressure_fields(
    eta=eta,
    eta_v=eta_v,
    delp=numpy_state.delp[slice_3d],
    ps=numpy_state.ps[slice_2d],
    pe=numpy_state.pe[slice_3d],
    peln=numpy_state.peln[slice_3d],
    pk=numpy_state.pk[slice_3d],
    pkz=numpy_state.pkz[slice_3d],
    ak=utils.asarray(grid_data.ak.data),
    bk=utils.asarray(grid_data.bk.data),
    ptop=grid_data.ptop,
)
alpha = 0
# Initialize dry atmosphere
numpy_state.qvapor[:] = 3.0e-6
numpy_state.qliquid[:] = 3.0e-6
numpy_state.qice[:] = 3.0e-6
numpy_state.qrain[:] = 3.0e-6
numpy_state.qsnow[:] = 3.0e-6
numpy_state.qgraupel[:] = 3.0e-6
numpy_state.qo3mr[:] = 3.0e-6
numpy_state.qsgs_tke[:] = 3.0e-6
numpy_state.qcld[:] = 3.0e-6
numpy_state.q_con[:] = 3.0e-6
numpy_state.u[:] = 0.0
numpy_state.v[:] = 0.0
if not hydrostatic:
    numpy_state.w[:] = 0.0

numpy_state.phis[:] = 0.0
print(numpy_state.ps.shape)

3 28 3 28
24 24 79
(31, 31)


In [34]:
def hydro_eq(
    km,
    is_,
    ie,
    js,
    je,
    ps,
    hs,
    drym,
    delp,
    ak,
    bk,
    pt,
    delz,
    area,
    ng,
    mountain,
    hydrostatic,
    hybrid_z,
    comm,
):
    # ndsl_log.info('Initializing ATM hydrostatically')
    # ndsl_log.info('Initializing Earth')

    gz = np.empty((ie, km+1))
    ph = np.empty((ie, km+1))
    print(ph.shape)

    # Given p1 and z1 (250mb, 10km)
    p1 = 25000.0
    z1 = 10.0e3 * constants.GRAV
    t1 = 200.0
    t0 = 300.0  # sea-level temp.
    a0 = (t1 - t0) / z1 * 0.5
    c0 = t0 / a0

    if hybrid_z:
        ptop = 100.0  # *** hardwired model top ***
    else:
        ptop = ak[0]

    ztop = z1 + (constants.RDGAS * t1) * np.log(p1 / ptop)
    # ndsl_log.info(f'ZTOP is computed as {ztop / constants.GRAV * 1.E-3}')

    if mountain:
        raise NotImplementedError("hydro_eq: Mountain is not implemented")
        """
        mslp = 100917.4
        for j in range(js, je):
            for i in range(is_, ie):
                ps[i, j] = mslp * np.exp(
                    -1./(a0 * constants.RDGAS) * hs[i, j] / (hs[i, j] + c0)
                )

        # this is the issue with Mountain:
        psm = g_sum(
            comm, ps[is_:ie, js:je], is_, ie, js, je, ng, area, 1, True
        )

        dps = drym - psm
        # ndsl_log.info(f'Computed mean ps={psm}')
        # ndsl_log.info(f'Correction delta-ps={dps}')
        """
    else:
        mslp = drym  # 1000.E2
        ps[is_:ie, js:je] = mslp
        dps = 0.0

    for j in range(js, je):
        for i in range(is_, ie):
            ps[i, j] = ps[i, j] + dps
            gz[i, 0] = ztop
            gz[i, km] = hs[i, j]
            ph[i, 0] = ptop
            ph[i, km] = ps[i, j]

        if hybrid_z:
            # ---------------
            # Hybrid Z
            # ---------------
            for k in range(km-1, 0, -1):  # k=km,2,-1
                for i in range(is_, ie):
                    gz[i, k] = gz[i, k + 1] - delz[i, j, k] * constants.GRAV
            # Correct delz at the top:
            for i in range(is_, ie):
                delz[i, j, 0] = (gz[i, 1] - ztop) / constants.GRAV

            for k in range(1, km):  # k=2,km
                for i in range(is_, ie):
                    if gz[i, k] >= z1:
                        # Isothermal
                        ph[i, k] = ptop * np.exp(
                            (gz[i, 0] - gz[i, k]) / (constants.RDGAS * t1)
                        )
                    else:
                        # Constant lapse rate region (troposphere)
                        ph[i, k] = ps[i, j] * np.exp(
                            -1.0
                            / (a0 * constants.RDGAS)
                            * (gz[i, k] - hs[i, j])
                            / (gz[i, k] - hs[i, j] + c0)
                        )
        else:
            # ---------------
            # Hybrid sigma-p
            # ---------------
            for k in range(1, km+1):  # do k=2,km+1
                for i in range(is_, ie):
                    ph[i, k] = ak[k] + bk[k] * ps[i, j]

            for k in range(km-1, 0, -1):  # k=km,2,-1
                for i in range(is_, ie):
                    if ph[i, k] <= p1:
                        gz[i, k] = gz[i, k + 1] + (constants.RDGAS * t1) * np.log(
                            ph[i, k + 1] / ph[i, k]
                        )
                    else:
                        # Constant lapse rate region (troposphere)
                        gz[i, k] = (
                            c0
                            / (1 + a0 * constants.RDGAS * np.log(ph[i, k] / ps[i, j]))
                            + hs[i, j]
                            - c0
                        )  # model top
            for i in range(is_, ie):
                if ph[i, 0] <= p1:
                    gz[i, 0] = gz[i, 1] + (constants.RDGAS * t1) * np.log(
                        ph[i, 1] / ph[i, 0]
                    )
                else:
                    gz[i, 0] = (hs[i, j] + c0) / (ph[i, 0] / ps[i, j]) ** (
                        a0 * constants.RDGAS
                    ) - c0
            if not hydrostatic:
                for k in range(km):
                    for i in range(is_, ie):
                        delz[i, j, k] = (gz[i, k + 1] - gz[i, k]) / constants.GRAV

        # Convert geopotential to Temperature
        for k in range(km):
            for i in range(is_, ie):
                pt[i, j, k] = (gz[i, k] - gz[i, k + 1]) / (
                    constants.RDGAS * (np.log(ph[i, k + 1] / ph[i, k]))
                )
                pt[i, j, k] = max(t1, pt[i, j, k])
                delp[i, j, k] = ph[i, k + 1] - ph[i, k]
        if j == js:
            i = is_
            for k in range(km):
                ndsl_log.info(
                    f"{k}, {pt[i, j, k]}, {gz[i, k+1]}, {(gz[i, k]-gz[i, k+1])}, {ph[i, k]}"
                )

In [35]:
hydro_eq(
    nz,
    isc,
    iec,
    jsc,
    jec,
    numpy_state.ps[:],
    numpy_state.phis[:],
    1.0e5,
    numpy_state.delp[:],
    grid_data.ak.data[:],
    grid_data.bk.data[:],
    numpy_state.pt[:],
    numpy_state.delz[:],
    grid_data.area.data[:],
    NHALO,
    False,
    hydrostatic,
    hybrid_z,
    comm,
)

(28, 80)
2026-03-24 18:46:20|INFO|rank 0|ndsl.logging:0, 200.0, 308956.91185718955, 44098.03442591196, 300.0
2026-03-24 18:46:20|INFO|rank 0|ndsl.logging:1, 200.0, 281395.66329888214, 27561.248558307416, 646.7159
2026-03-24 18:46:20|INFO|rank 0|ndsl.logging:2, 200.00000000000023, 261848.6741096665, 19546.989189215645, 1045.222
2026-03-24 18:46:20|INFO|rank 0|ndsl.logging:3, 200.00000000000017, 247151.67196655957, 14697.002143106918, 1469.188
2026-03-24 18:46:20|INFO|rank 0|ndsl.logging:4, 200.00000000000003, 235487.3943303705, 11664.277636189072, 1897.829
2026-03-24 18:46:20|INFO|rank 0|ndsl.logging:5, 200.00000000000023, 225767.1600259179, 9720.234304452606, 2325.385
2026-03-24 18:46:20|INFO|rank 0|ndsl.logging:6, 200.0, 217314.78353188868, 8452.376494029217, 2754.396
2026-03-24 18:46:20|INFO|rank 0|ndsl.logging:7, 200.0, 209630.82247519473, 7683.961056693952, 3191.294
2026-03-24 18:46:20|INFO|rank 0|ndsl.logging:8, 200.0000000000002, 202432.7291792973, 7198.093295897415, 3648.332
202

2026-03-24 18:46:21|INFO|rank 0|ndsl.logging:72, 296.58428749008647, 2974.6143594639143, 769.0364910052158, 95719.2216
2026-03-24 18:46:21|INFO|rank 0|ndsl.logging:73, 297.3253235965561, 2282.8495139000006, 691.7648455639137, 96587.7888
2026-03-24 18:46:21|INFO|rank 0|ndsl.logging:74, 297.98845677594744, 1668.9255958526628, 613.9239180473378, 97373.843
2026-03-24 18:46:21|INFO|rank 0|ndsl.logging:75, 298.57289583051664, 1133.311130723916, 535.6144651287468, 98075.2326
2026-03-24 18:46:21|INFO|rank 0|ndsl.logging:76, 299.07797827790455, 676.3805050813826, 456.9306256425334, 98690.0717
2026-03-24 18:46:21|INFO|rank 0|ndsl.logging:77, 299.50309077839216, 298.565667877323, 377.8148372040596, 99216.741
2026-03-24 18:46:21|INFO|rank 0|ndsl.logging:78, 299.8477738738773, 0.0, 298.565667877323, 99653.7191618097


In [39]:
type(grid_data.ak.data[:])

numpy.ndarray